# Contenido de `datos_unzipped/`

Recorrido por cada carpeta del dataset para saber **qué tipo de información contiene**: cuántos archivos hay, en qué formato, cómo se nombran y qué columnas tienen.

El notebook solo lee lo necesario: de los CSV de sEMG (10 GB) lee las primeras filas, no los archivos completos.

**Contenido**
1. Vista general de las carpetas
2. `Data/`: señales sEMG
3. `Labels/`: etiquetas de actividad en frames de sEMG
4. `Labels 2/`: etiquetas de actividad en frames de captura de movimiento (MVNX)
5. `Subjs_info.xlsx`: datos de los participantes
6. `m_files/`: scripts de MATLAB de los autores
7. Resumen

In [1]:
from pathlib import Path
import re

import pandas as pd

# El notebook vive en notebooks/; los datos están en la raíz del repo
REPO = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
ROOT = REPO / "datos_unzipped"
assert ROOT.exists(), f"No se encontró {ROOT}"

pd.set_option("display.max_columns", 50)


def human_size(n_bytes):
    for unit in ("B", "KB", "MB", "GB"):
        if n_bytes < 1024:
            return f"{n_bytes:.1f} {unit}"
        n_bytes /= 1024
    return f"{n_bytes:.1f} TB"


def files_in(folder):
    # Ignora archivos ocultos de macOS (.DS_Store)
    return sorted(p for p in folder.rglob("*") if p.is_file() and not p.name.startswith("."))

## 1. Vista general de las carpetas

In [2]:
rows = []
for item in sorted(ROOT.iterdir()):
    if item.name.startswith("."):
        continue
    if item.is_dir():
        fs = files_in(item)
        subdirs = sorted(d.name for d in item.iterdir() if d.is_dir())
        exts = pd.Series([f.suffix for f in fs]).value_counts().to_dict()
        rows.append({
            "elemento": item.name + "/",
            "subcarpetas": len(subdirs),
            "archivos": len(fs),
            "extensiones": exts,
            "tamaño": human_size(sum(f.stat().st_size for f in fs)),
            "ejemplo": fs[0].relative_to(item).as_posix() if fs else "",
        })
    else:
        rows.append({"elemento": item.name, "subcarpetas": 0, "archivos": 1,
                     "extensiones": {item.suffix: 1}, "tamaño": human_size(item.stat().st_size),
                     "ejemplo": item.name})

overview = pd.DataFrame(rows)
overview

,elemento,subcarpetas,archivos,extensiones,tamaño,ejemplo
0,Data/,14,679,{'.csv': 679},10.2 GB,Subj_01/Subj_01_Isokin_L_02kg_40bpm_EMG_data.csv
1,Labels/,14,679,{'.csv': 679},179.2 KB,Subj_01/Subj_01_Isokin_L_02kg_40bpm_EMG_labels...
2,Labels 2/,14,517,{'.csv': 517},129.3 KB,Subj_01/Subj_01_Isokin_L_02kg_40bpm_mvnx_label...
3,Subjs_info.xlsx,0,1,{'.xlsx': 1},9.4 KB,Subjs_info.xlsx
4,m_files/,0,2,{'.m': 2},9.0 KB,EMG_processing.m
5,mvnx_files/,14,517,{'.mvnx': 517},52.8 GB,Subj_01/Subj_01_Isokin_L_02kg_40bpm.mvnx


`Data/`, `Labels/` y `Labels 2/` están organizadas igual: **una subcarpeta por sujeto** (`Subj_01` … `Subj_14`) y un CSV por prueba (*trial*). El nombre de cada archivo describe la prueba, así que lo primero es separarlo en columnas.

In [3]:
# Ejemplos:
#   Subj_01_Isokin_L_02kg_40bpm_EMG_data.csv
#   Subj_01_MMH_Bim_L_08kg_Trial_2_EMG_labels.csv
#   Subj_01_MMH_Bim_08kg_Trial_2_mvnx_labels.csv   (en MVNX el bimanual no tiene lado)
#   Subj_12_MVC_All_L_EMG_range.csv
PATTERN = re.compile(
    r"Subj_(?P<sujeto>\d+)_"
    r"(?P<tarea>Isokin|MMH_Bim|MMH_One|MVC_All|MVC_Isokin|MVC_MMH_Bim|MVC_MMH_One)"
    r"(?:_(?P<lado>[LR]))?"
    r"(?:_(?P<carga_kg>\d+)kg)?"
    r"(?:_(?P<condicion>St|\d+bpm|Trial_\d))?"
    r"_(?P<tipo>EMG_data|EMG_labels|EMG_range|mvnx_labels)\.csv$"
)


def parse_names(folder):
    records = []
    for f in files_in(folder):
        m = PATTERN.match(f.name)
        assert m, f"Nombre no reconocido: {f.name}"
        records.append({**m.groupdict(), "archivo": f.name, "bytes": f.stat().st_size, "ruta": f})
    df = pd.DataFrame(records)
    df["sujeto"] = df["sujeto"].astype(int)
    df["carga_kg"] = pd.to_numeric(df["carga_kg"])
    return df

## 2. `Data/`: señales sEMG

Electromiografía de superficie (**sEMG**) de 8 canales, un CSV por prueba.

Tipos de tarea (según el nombre del archivo):

| Tarea | Qué es | Condición |
|---|---|---|
| `Isokin` | Movimiento isocinético con un brazo (`L`/`R`), 2 y 4 kg | `St` (estático) o ritmo de 40/60/80 bpm |
| `MMH_Bim` | *Manual Material Handling* bimanual: levantar, cargar y colocar un objeto con ambas manos, 2/8/12 kg | `Trial_1..3` |
| `MMH_One` | *Manual Material Handling* con una mano (`L`/`R`), 2 y 4 kg | `Trial_1..3` |
| `MVC_*` | Contracción voluntaria máxima, usada como referencia para normalizar la señal a %MVC | – |

In [4]:
data = parse_names(ROOT / "Data")
print(f"{len(data)} archivos, {human_size(data.bytes.sum())}, {data.sujeto.nunique()} sujetos")

# Pruebas por tarea y lado
data.pivot_table(index="tarea", columns="lado", values="archivo", aggfunc="count", fill_value=0, margins=True, margins_name="total")

679 archivos, 10.2 GB, 14 sujetos


lado,L,R,total
tarea,,,
Isokin,111,112,223
MMH_Bim,126,126,252
MMH_One,84,84,168
MVC_All,12,12,24
MVC_Isokin,2,2,4
MVC_MMH_Bim,2,2,4
MVC_MMH_One,2,2,4
total,339,340,679


In [5]:
# Archivos por sujeto y tarea: permite ver si a algún sujeto le faltan pruebas
data.pivot_table(index="sujeto", columns="tarea", values="archivo", aggfunc="count", fill_value=0, margins=True, margins_name="total")

tarea,Isokin,MMH_Bim,MMH_One,MVC_All,MVC_Isokin,MVC_MMH_Bim,MVC_MMH_One,total
sujeto,,,,,,,,
1,16,18,12,0,2,2,2,52
2,16,18,12,2,0,0,0,48
3,16,18,12,2,0,0,0,48
4,16,18,12,2,0,0,0,48
5,16,18,12,2,0,0,0,48
6,16,18,12,2,0,0,0,48
7,16,18,12,2,0,0,0,48
8,16,18,12,2,0,0,0,48
9,16,18,12,2,0,0,0,48


In [6]:
# Cargas y condiciones que aparecen en cada tarea
data.groupby("tarea").agg(
    cargas_kg=("carga_kg", lambda s: sorted(s.dropna().astype(int).unique())),
    condiciones=("condicion", lambda s: sorted(s.dropna().unique())),
    tamaño_medio=("bytes", lambda s: human_size(s.mean())),
)

,cargas_kg,condiciones,tamaño_medio
tarea,,,
Isokin,"[2, 4]","[40bpm, 60bpm, 80bpm, St]",12.6 MB
MMH_Bim,"[2, 8, 12]","[Trial_1, Trial_2, Trial_3]",17.4 MB
MMH_One,"[2, 4]","[Trial_1, Trial_2, Trial_3]",17.5 MB
MVC_All,[],[],7.4 MB
MVC_Isokin,[],[],7.5 MB
MVC_MMH_Bim,[],[],8.9 MB
MVC_MMH_One,[],[],8.6 MB


### Contenido de un archivo sEMG

Solo leemos las primeras filas de un archivo de ejemplo.

In [7]:
sample_path = data.query("tarea == 'MMH_Bim'").ruta.iloc[0]
emg = pd.read_csv(sample_path, nrows=5000)
print(sample_path.name, "->", emg.shape[1], "columnas")
emg.head()

Subj_01_MMH_Bim_L_02kg_Trial_1_EMG_data.csv -> 25 columnas


,Time,Raw_V_1,Raw_V_2,Raw_V_3,Raw_V_4,Raw_V_5,Raw_V_6,Raw_V_7,Raw_V_8,Norm_V_1,Norm_V_2,Norm_V_3,Norm_V_4,Norm_V_5,Norm_V_6,Norm_V_7,Norm_V_8,Norm_RMS_1,Norm_RMS_2,Norm_RMS_3,Norm_RMS_4,Norm_RMS_5,Norm_RMS_6,Norm_RMS_7,Norm_RMS_8
0,1.592248e+12,-0.005415,-0.001652,-0.001140,0.000690,-0.001871,-0.008812,0.002267,-0.000679,10.655998,8.064956,8.143451,8.028745,5.232562,3.988142,5.408433,4.715932,13.062016,3.337926,2.949881,3.553770,2.929462,1.632712,2.486120,1.919630
1,1.592248e+12,-0.005545,-0.001749,-0.001232,0.000596,-0.001964,-0.008911,0.002166,-0.000771,27.418658,17.205715,15.948248,17.154502,10.695209,8.880229,12.240876,9.605845,13.073258,3.336219,2.939926,3.540803,2.925080,1.627104,2.486181,1.912574
2,1.592248e+12,-0.005438,-0.001650,-0.001138,0.000685,-0.001876,-0.008813,0.002262,-0.000684,28.340562,14.930683,13.206531,15.980085,10.176868,7.707219,11.057866,9.119564,13.072329,3.327468,2.931676,3.527126,2.919472,1.621196,2.479421,1.905118
3,1.592248e+12,-0.005552,-0.001745,-0.001225,0.000595,-0.001973,-0.008904,0.002170,-0.000777,26.995481,12.720571,9.784060,13.531525,9.629063,6.114085,9.015063,8.107436,13.060031,3.316079,2.920369,3.513628,2.912001,1.614913,2.470127,1.899253
4,1.592248e+12,-0.005440,-0.001648,-0.001128,0.000693,-0.001880,-0.008806,0.002270,-0.000680,24.494676,11.023952,7.103755,11.315801,8.876878,4.796085,7.229193,6.570874,13.028556,3.304596,2.909568,3.500275,2.901614,1.608818,2.460900,1.892854


In [8]:
# Las columnas se agrupan por prefijo
groups = pd.Series(emg.columns).str.replace(r"_\d+$", "", regex=True).value_counts(sort=False)
print(groups.to_string())

print(f"\nTime es una marca Unix en ms. Inicio de la grabación: {pd.to_datetime(emg['Time'].iloc[0], unit='ms')}")


def sampling_rate(path):
    # Igual que Process.m: número de muestras / duración total
    t = pd.read_csv(path, usecols=["Time"])["Time"]
    return len(t) / ((t.iloc[-1] - t.iloc[0]) / 1000)


# La frecuencia NO es fija: cambia entre sujetos y sesiones (se leen solo unos archivos de ejemplo)
examples = data.query("tarea == 'Isokin' and lado == 'L' and carga_kg == 2 and condicion == '40bpm'").head(6)
examples.assign(frecuencia_Hz=[round(sampling_rate(p), 1) for p in examples.ruta])[["archivo", "frecuencia_Hz"]]

Time        1
Raw_V       8
Norm_V      8
Norm_RMS    8

Time es una marca Unix en ms. Inicio de la grabación: 2020-06-15 19:03:04.147000


,archivo,frecuencia_Hz
0,Subj_01_Isokin_L_02kg_40bpm_EMG_data.csv,692.8
52,Subj_02_Isokin_L_02kg_40bpm_EMG_data.csv,523.6
100,Subj_03_Isokin_L_02kg_40bpm_EMG_data.csv,532.0
148,Subj_04_Isokin_L_02kg_40bpm_EMG_data.csv,467.3
196,Subj_05_Isokin_L_02kg_40bpm_EMG_data.csv,438.6
244,Subj_06_Isokin_L_02kg_40bpm_EMG_data.csv,521.5


| Columna | Qué contiene | Unidad |
|---|---|---|
| `Time` | Marca de tiempo Unix | ms |
| `Raw_V_1..8` | Voltaje crudo de cada uno de los 8 canales (electrodos) | V |
| `Norm_V_1..8` | Señal filtrada y rectificada, normalizada respecto a la MVC | %MVC |
| `Norm_RMS_1..8` | Envolvente RMS (ventana de 250 muestras) normalizada | %MVC |

**Frecuencia de muestreo variable:** dentro de un archivo el muestreo es regular, pero la frecuencia cambia entre archivos (de ~370 a ~1006 Hz según el EDA). Para convertir frames a segundos hay que calcular la frecuencia de cada archivo; para modelar conviene remuestrear a una frecuencia común.

Los archivos `MVC_*` solo tienen `Raw_V` y `Rect_V` (sin normalizar), porque ellos **son** la referencia de la normalización:

In [9]:
mvc_path = data.query("tarea.str.startswith('MVC')", engine="python").ruta.iloc[0]
print(mvc_path.name)
print(list(pd.read_csv(mvc_path, nrows=1).columns))

Subj_01_MVC_Isokin_L_EMG_data.csv
['Time', 'Raw_V_1', 'Raw_V_2', 'Raw_V_3', 'Raw_V_4', 'Raw_V_5', 'Raw_V_6', 'Raw_V_7', 'Raw_V_8', 'Rect_V_1', 'Rect_V_2', 'Rect_V_3', 'Rect_V_4', 'Rect_V_5', 'Rect_V_6', 'Rect_V_7', 'Rect_V_8']


## 3. `Labels/`: etiquetas de actividad (frames de sEMG)

Hay **un archivo por cada archivo de `Data/`**, con el mismo nombre pero terminado en `_EMG_labels.csv` o `_EMG_range.csv`:

- `*_EMG_labels.csv`: segmentos de actividad (`Start_Frame`, `End_Frame`, `Label`). Los frames son **índices de fila del CSV de sEMG** correspondiente.
- `*_EMG_range.csv` (solo en pruebas MVC): el rango de frames usado para calcular el valor máximo de referencia.

In [10]:
labels = parse_names(ROOT / "Labels")
print(labels.tipo.value_counts().to_string())

# ¿Cada archivo de Data/ tiene su archivo de etiquetas?
key = lambda s: s.str.replace(r"_EMG_(data|labels|range)\.csv$", "", regex=True)
missing = set(key(data.archivo)) ^ set(key(labels.archivo))
print("\nArchivos sin pareja entre Data/ y Labels/:", missing or "ninguno")

tipo
EMG_labels    643
EMG_range      36

Archivos sin pareja entre Data/ y Labels/: ninguno


In [11]:
print("Ejemplo de etiquetas:")
display(pd.read_csv(labels.query("tipo == 'EMG_labels'").ruta.iloc[0]).head(8))

print("Ejemplo de rango MVC:")
display(pd.read_csv(labels.query("tipo == 'EMG_range'").ruta.iloc[0]))

Ejemplo de etiquetas:


,Start_Frame,End_Frame,Label
0,14402,16884,N
1,16887,17898,LT
2,17901,19367,PT
3,19370,23441,N
4,23443,24431,LT
5,24433,25698,PT
6,25701,30721,N
7,30724,31644,LT


Ejemplo de rango MVC:


,Start_Frame,End_Frame
0,23000,25000


In [12]:
# Todas las etiquetas juntas (son archivos pequeños)
seg = pd.concat(
    [pd.read_csv(r.ruta).assign(sujeto=r.sujeto, tarea=r.tarea) for r in labels.query("tipo == 'EMG_labels'").itertuples()],
    ignore_index=True,
)
seg["duracion_frames"] = seg.End_Frame - seg.Start_Frame
print(f"{len(seg)} segmentos etiquetados en {labels.tipo.eq('EMG_labels').sum()} archivos")

# Qué clases aparecen en cada tarea
seg.pivot_table(index="Label", columns="tarea", values="Start_Frame", aggfunc="count", fill_value=0, margins=True, margins_name="total")

10612 segmentos etiquetados en 643 archivos


tarea,Isokin,MMH_Bim,MMH_One,total
Label,,,,
K,56,252,168,476
LF,0,706,492,1198
LT,884,484,330,1698
N,1107,1434,989,3530
PF,0,484,330,814
PT,884,706,492,2082
W,0,484,330,814
total,2931,4550,3131,10612


Las 7 clases de actividad (definiciones del artículo que usa el dataset, *Sensors* 25(21), 6705, 2025):

| Código | Actividad | | Código | Actividad |
|---|---|---|---|---|
| `N` | N-pose (de pie, neutral) | | `LT` | Levantar desde la mesa |
| `LF` | Levantar desde el piso | | `W` | Caminar / cargar |
| `K` | Mantener el objeto levantado | | `PF` | Colocar en el piso |
| `PT` | Colocar en la mesa | | | |

Las pruebas `Isokin` solo tienen `N`, `LT`, `PT` (y `K` en la condición estática); las pruebas `MMH` tienen las 7 clases.

## 4. `Labels 2/`: etiquetas de actividad (frames de captura de movimiento)

Tienen el mismo formato (`Start_Frame`, `End_Frame`, `Label`), pero los archivos terminan en `_mvnx_labels.csv`: los frames se refieren a las grabaciones de **captura de movimiento Xsens (MVNX)**, no al sEMG.

Diferencias con `Labels/`:
- No hay pruebas MVC (la captura de movimiento no se usó en ellas).
- En `MMH_Bim` no aparece el lado: hay **una sola** grabación de movimiento para las dos señales sEMG (`Bim_L` y `Bim_R`).

In [13]:
labels2 = parse_names(ROOT / "Labels 2")
print(f"{len(labels2)} archivos ({labels2.tipo.unique().tolist()})")
# MMH_Bim no tiene lado en MVNX: se muestra como "ambos"
labels2.assign(lado=labels2.lado.fillna("ambos")).pivot_table(
    index="tarea", columns="lado", values="archivo", aggfunc="count", fill_value=0, margins=True, margins_name="total")

517 archivos (['mvnx_labels'])


lado,L,R,ambos,total
tarea,,,,
Isokin,111,112,0,223
MMH_Bim,0,0,126,126
MMH_One,84,84,0,168
total,195,196,126,517


In [14]:
seg2 = pd.concat(
    [pd.read_csv(r.ruta).assign(sujeto=r.sujeto, tarea=r.tarea) for r in labels2.itertuples()],
    ignore_index=True,
)
print(f"{len(seg2)} segmentos; clases: {sorted(seg2.Label.unique())}")
pd.read_csv(labels2.ruta.iloc[0]).head(8)

8337 segmentos; clases: ['K', 'LF', 'LT', 'N', 'PF', 'PT', 'W']


,Start_Frame,End_Frame,Label
0,2846,3706,N
1,3707,4057,LT
2,4058,4566,PT
3,4567,5977,N
4,5978,6320,LT
5,6321,6759,PT
6,6760,8499,N
7,8500,8819,LT


**¿Por qué los números de frame son distintos?** Cada modalidad tiene su propia frecuencia de muestreo. Si comparamos la misma prueba en ambas carpetas, el cociente de las duraciones de los segmentos debería ser aproximadamente `frecuencia sEMG de ese archivo / frecuencia MVNX (240 Hz)`.

In [15]:
# Misma prueba (un Isokin, que tiene lado en ambas carpetas) en las dos referencias
name = labels2.query("tarea == 'Isokin'").archivo.iloc[0].replace("_mvnx_labels.csv", "")
a = pd.read_csv(ROOT / "Labels" / name[:7] / f"{name}_EMG_labels.csv")
b = pd.read_csv(ROOT / "Labels 2" / name[:7] / f"{name}_mvnx_labels.csv")

cmp = pd.DataFrame({
    "Label": a.Label,
    "frames_sEMG": a.End_Frame - a.Start_Frame,
    "frames_MVNX": b.End_Frame - b.Start_Frame,
})
cmp["cociente"] = (cmp.frames_sEMG / cmp.frames_MVNX).round(2)
fs_emg = sampling_rate(ROOT / "Data" / name[:7] / f"{name}_EMG_data.csv")
print(name)
print(f"Cociente mediano: {cmp.cociente.median():.2f}  (≈ {fs_emg:.1f} Hz sEMG de este archivo / 240 Hz MVNX = {fs_emg / 240:.2f})")
cmp.head(8)

Subj_01_Isokin_L_02kg_40bpm
Cociente mediano: 2.89  (≈ 692.8 Hz sEMG de este archivo / 240 Hz MVNX = 2.89)


,Label,frames_sEMG,frames_MVNX,cociente
0,N,2482,860,2.89
1,LT,1011,350,2.89
2,PT,1466,508,2.89
3,N,4071,1410,2.89
4,LT,988,342,2.89
5,PT,1265,438,2.89
6,N,5020,1739,2.89
7,LT,920,319,2.88


> **Nota:** las grabaciones MVNX (archivos `.mvnx`, XML de Xsens, ~56 GB) **no están** en `datos_unzipped/`. En `Labels 2/` solo están sus etiquetas.

## 5. `Subjs_info.xlsx`: datos de los participantes

Una fila por sujeto con datos demográficos y **antropométricos** (medidas corporales en cm, masa en kg).

In [16]:
subjs = pd.read_excel(ROOT / "Subjs_info.xlsx")
display(subjs)
subjs.describe().T[["mean", "min", "max"]].round(1)

,ID,Sex,Age,Body_mass,Body_height,Foot_length,Shoulder_height,Shoulder_width,Elbow_span,Wrist_span,Arm_span,Hip_height,Hip_width,Knee_height,Ankle_height,Extra_shoe_thickeness
0,1,M,38,70,181,29.5,151.0,44,93.7,142.8,182,106.0,34.0,52.0,10,1
1,2,M,37,79,184,30.0,156.0,39,95.0,141.0,183,103.0,37.0,59.0,13,3
2,3,F,35,58,166,26.0,140.0,38,83.0,122.0,161,97.0,28.0,52.0,14,8
3,4,M,28,78,187,32.0,152.0,41,96.0,146.0,186,100.0,34.0,52.0,14,3
4,5,M,25,72,180,30.0,154.0,45,89.0,144.0,180,105.0,33.0,54.0,13,1
5,6,M,28,72,176,29.5,150.0,45,88.0,140.0,173,100.0,34.0,56.0,15,1
6,7,M,27,70,178,29.0,148.0,45,90.0,137.0,175,96.0,35.0,52.0,13,1
7,8,M,28,72,178,30.0,144.0,42,91.0,120.0,174,96.0,33.0,46.0,15,1
8,9,M,27,68,181,29.0,149.0,45,94.0,148.0,185,108.0,31.0,53.0,13,1
9,10,M,37,73,174,30.0,143.0,38,82.0,123.0,164,98.0,33.0,53.0,16,2


,mean,min,max
ID,7.5,1.0,14.0
Age,30.6,25.0,38.0
Body_mass,71.6,58.0,90.0
Body_height,178.6,166.0,187.0
Foot_length,29.3,25.0,32.0
Shoulder_height,148.7,140.0,156.0
Shoulder_width,41.6,32.0,46.0
Elbow_span,88.8,76.0,96.0
Wrist_span,133.8,103.0,148.0
Arm_span,176.6,161.0,188.0


## 6. `m_files/`: scripts de MATLAB

Código de los autores del dataset para procesar el sEMG crudo. Es útil para entender cómo se generaron las columnas `Norm_V` y `Norm_RMS`.

In [17]:
for f in files_in(ROOT / "m_files"):
    lines = f.read_text(errors="replace").splitlines()
    comments = [l.strip("% ").rstrip() for l in lines if l.strip().startswith("%")][:6]
    print(f"=== {f.name} ({len(lines)} líneas)")
    print(lines[0] if lines[0].startswith("function") else "\n".join(comments))
    print()

=== EMG_processing.m (107 líneas)
This file includes the parsing and processing of the EMG raw data. This
file requires setting the path where data have been downloaded from
Zenodo. If unchanged, this file processes the whole dataset. The lines
where specific subjects and trials can be selected are commented along
with an example of selection.
The following lines must be edited to set the basepath of the EMG data and EMG labels folders.

=== Process.m (44 líneas)
function Data = Process(Data, subject_num, activity_set, file_path, m)



- `EMG_processing.m`: script principal. Recorre sujetos y pruebas, lee `EMG_Data` y `EMG_Labels` y llama a `Process`.
- `Process.m`: procesamiento de cada prueba:
  1. filtro *notch* de 50 Hz (interferencia eléctrica),
  2. filtro pasa-banda Butterworth desde 20 Hz,
  3. media móvil y rectificación (`abs`),
  4. envolvente RMS de 250 muestras,
  5. normalización respecto a la MVC (%MVC).

## 7. Resumen

In [18]:
summary = pd.DataFrame([
    {"carpeta": "Data/", "contenido": "Señales sEMG de 8 canales (crudas + normalizadas %MVC)", "unidad": "1 CSV por prueba",
     "archivos": len(data), "tamaño": human_size(data.bytes.sum())},
    {"carpeta": "Labels/", "contenido": "Segmentos de actividad en frames de sEMG (+ rangos MVC)", "unidad": "1 CSV por archivo de Data/",
     "archivos": len(labels), "tamaño": human_size(labels.bytes.sum())},
    {"carpeta": "Labels 2/", "contenido": "Segmentos de actividad en frames de captura de movimiento (MVNX)", "unidad": "1 CSV por grabación MVNX",
     "archivos": len(labels2), "tamaño": human_size(labels2.bytes.sum())},
    {"carpeta": "Subjs_info.xlsx", "contenido": "Demografía y antropometría de los 14 sujetos", "unidad": "1 fila por sujeto",
     "archivos": 1, "tamaño": human_size((ROOT / "Subjs_info.xlsx").stat().st_size)},
    {"carpeta": "m_files/", "contenido": "Scripts MATLAB de filtrado y normalización del sEMG", "unidad": ".m",
     "archivos": len(files_in(ROOT / "m_files")), "tamaño": human_size(sum(f.stat().st_size for f in files_in(ROOT / "m_files")))},
])
summary

,carpeta,contenido,unidad,archivos,tamaño
0,Data/,Señales sEMG de 8 canales (crudas + normalizad...,1 CSV por prueba,679,10.2 GB
1,Labels/,Segmentos de actividad en frames de sEMG (+ ra...,1 CSV por archivo de Data/,679,179.2 KB
2,Labels 2/,Segmentos de actividad en frames de captura de...,1 CSV por grabación MVNX,517,129.3 KB
3,Subjs_info.xlsx,Demografía y antropometría de los 14 sujetos,1 fila por sujeto,1,9.4 KB
4,m_files/,Scripts MATLAB de filtrado y normalización del...,.m,2,9.0 KB
